# 01 — Análisis Exploratorio de Datos (EDA)
**Proyecto:** OpenFarma — Sistema de Alerta Temprana de Desabastecimiento Farmacéutico  
**Fuentes:** CUM activo (datos.gov.co · i7cb-raxc) + Alertas INVIMA (PDF mensual)  
**Concurso:** Datos al Ecosistema 2026 — IA para Colombia

Este notebook explora las dos fuentes de datos principales para entender la distribución del mercado farmacéutico colombiano y el histórico de alertas del INVIMA.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 120

DB_PATH = Path('../backend/openfarma.db')
conn = sqlite3.connect(DB_PATH)
print(f'Conectado a {DB_PATH} ({DB_PATH.stat().st_size / 1_000_000:.1f} MB)')

## 1. Visión general del CUM
El Catálogo Único de Medicamentos (CUM) es la fuente oficial del INVIMA. Contiene todos los medicamentos con registro sanitario activo en Colombia.

In [ ]:
cum = pd.read_sql("""
    SELECT expediente_cum, consecutivo_cum, nombre_comercial_norm,
           principios_dci, tipo_formula, forma_normalizada,
           via_normalizada, atc_normalizado, estado_cum, estado_registro,
           titular_registro, dosis_total_mg, concentracion_mg_ml
    FROM cum_normalizado
""", conn)

print(f'Total registros CUM: {len(cum):,}')
print(f'Activos: {(cum.estado_cum == "Activo").sum():,}')
print(f'Inactivos: {(cum.estado_cum != "Activo").sum():,}')
print(f'\nTitulares únicos: {cum.titular_registro.nunique():,}')
print(f'Formas farmacéuticas únicas: {cum.forma_normalizada.nunique():,}')

In [ ]:
# Distribución por tipo de fórmula
tipo_counts = cum[cum.estado_cum == 'Activo']['tipo_formula'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(tipo_counts, labels=tipo_counts.index, autopct='%1.1f%%',
            colors=['#2563eb','#60a5fa','#93c5fd','#bfdbfe'], startangle=90)
axes[0].set_title('Distribución por tipo de fórmula\n(CUM activos)', fontsize=12)

# Estado del registro
estado_counts = cum[cum.estado_cum == 'Activo']['estado_registro'].value_counts().head(5)
estado_counts.plot(kind='bar', ax=axes[1], color='#2563eb', edgecolor='white')
axes[1].set_title('Estado del registro sanitario\n(CUM activos)', fontsize=12)
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/distribuciones.png', bbox_inches='tight')
plt.show()
print('Figura guardada en reports/figures/distribuciones.png')

## 2. Distribución por grupo ATC
La clasificación ATC (Anatómica, Terapéutica, Química) organiza los medicamentos por su acción farmacológica.

In [ ]:
cum_activos = cum[cum.estado_cum == 'Activo'].copy()
cum_activos['grupo_atc1'] = cum_activos['atc_normalizado'].str[:1]

ATC_LABELS = {
    'A': 'A - Digestivo/Metab.', 'B': 'B - Sangre', 'C': 'C - Cardiovascular',
    'D': 'D - Dermatológico', 'G': 'G - Genitourinario', 'H': 'H - Hormonal',
    'J': 'J - Antiinfeccioso', 'L': 'L - Antineoplásico', 'M': 'M - Músculo-esquelético',
    'N': 'N - Sistema nervioso', 'P': 'P - Antiparasitario', 'R': 'R - Respiratorio',
    'S': 'S - Órganos sensoriales', 'V': 'V - Varios'
}

atc_counts = cum_activos['grupo_atc1'].map(ATC_LABELS).value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
atc_counts.plot(kind='bar', ax=ax, color='#2563eb', edgecolor='white')
ax.set_title('Medicamentos activos por grupo ATC (primer nivel)', fontsize=13)
ax.set_xlabel('Grupo ATC')
ax.set_ylabel('Número de presentaciones')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 3. Concentración de mercado — Top titulares

In [ ]:
top_titulares = (
    cum_activos
    .groupby('titular_registro')
    .size()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
top_titulares.sort_values().plot(kind='barh', ax=ax, color='#2563eb', edgecolor='white')
ax.set_title('Top 15 titulares por número de presentaciones activas', fontsize=12)
ax.set_xlabel('Presentaciones activas')
plt.tight_layout()
plt.show()

# Índice de concentración
total = len(cum_activos)
top3_share = top_titulares.head(3).sum() / total * 100
top10_share = top_titulares.head(10).sum() / total * 100
print(f'Top 3 titulares concentran: {top3_share:.1f}% del mercado')
print(f'Top 10 titulares concentran: {top10_share:.1f}% del mercado')

## 4. Análisis histórico de alertas INVIMA
17 meses de alertas (enero 2025 – mayo 2026). Tres categorías: MONITORIZACION, NO_DESABASTECIMIENTO, NO_COMERCIALIZADO.

In [ ]:
invima = pd.read_sql("""
    SELECT mes, anio, principio_activo, forma, concentracion,
           estado, atc
    FROM invima_seguimiento
    ORDER BY anio, mes
""", conn)

invima['periodo'] = pd.to_datetime(
    invima['anio'].astype(str) + '-' + invima['mes'].astype(str).str.zfill(2) + '-01'
)

print(f'Total entradas INVIMA: {len(invima):,}')
print(f'Período: {invima.periodo.min().strftime("%b %Y")} – {invima.periodo.max().strftime("%b %Y")}')
print(f'\nPrincipios activos únicos bajo vigilancia: {invima.principio_activo.nunique():,}')
print(f'\nDistribución por estado:')
print(invima['estado'].value_counts())

In [ ]:
# Evolución temporal de alertas INVIMA
timeline = (
    invima.groupby(['periodo', 'estado'])
    .size()
    .unstack(fill_value=0)
)

colors = {
    'MONITORIZACION': '#f59e0b',
    'NO_DESABASTECIMIENTO': '#10b981',
    'NO_COMERCIALIZADO': '#6b7280',
    'EN_RIESGO': '#ef4444',
    'DESABASTECIDO': '#dc2626',
}

fig, ax = plt.subplots(figsize=(14, 5))
for col in timeline.columns:
    ax.plot(timeline.index, timeline[col], marker='o', markersize=4,
            label=col, color=colors.get(col, '#94a3b8'), linewidth=2)

ax.set_title('Evolución mensual de alertas INVIMA (ene 2025 – may 2026)', fontsize=13)
ax.set_xlabel('Mes')
ax.set_ylabel('Número de alertas')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../reports/figures/alertas_invima_timeline.png', bbox_inches='tight')
plt.show()
print('Figura guardada en reports/figures/alertas_invima_timeline.png')

## 5. Medicamentos con mayor historial de alertas

In [ ]:
# Top principios activos con más meses en MONITORIZACION o DESABASTECIDO
criticos = (
    invima[invima['estado'].isin(['MONITORIZACION', 'EN_RIESGO', 'DESABASTECIDO'])]
    .groupby('principio_activo')
    .size()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 7))
criticos.sort_values().plot(kind='barh', ax=ax, color='#f59e0b', edgecolor='white')
ax.set_title('Top 20 principios activos con más meses de alerta INVIMA\n(monitorización + riesgo + desabastecido)', fontsize=12)
ax.set_xlabel('Meses con alerta')
plt.tight_layout()
plt.show()

print('Top 10 con más historial de alertas:')
print(criticos.head(10).to_string())

In [ ]:
conn.close()
print('EDA completado. Figuras generadas en reports/figures/')